# Persistent TP4 sharded state trial
The exact vLLM 0.30.0 source exposes EngineCoreClient.save_sharded_state and ShardedStateLoader. This trial saves four rank files with tensor metadata and SHA256 hashes, then reloads the model and compares deterministic token IDs. It will refuse an insufficient /kaggle/working filesystem or interrupted save. No upload follows automatically.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess
base = Path('/kaggle/input/datasets/waqasm86')
sdk_input = base/'kaggle-vllm-sdk-v031-l4-acceptance'
wheel = sdk_input/'kaggle_vllm-0.3.1-py3-none-any.whl'
pip_wheel = sdk_input/'pip-25.2-py3-none-any.whl'
model = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
assert all(p.is_file() for p in (wheel,pip_wheel,model/'config.json'))
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == 'a14a0be361ea0ab876aa939743ad7259ac012b8c9410cc668899da00b875c27b'
python312='/usr/bin/python3.12'
assert subprocess.run([python312,'-c','import sys;assert sys.version_info[:2]==(3,12)']).returncode == 0
env=dict(os.environ)
env['PYTHONPATH']=os.pathsep.join(filter(None,(str(pip_wheel),env.get('PYTHONPATH',''))))
env.update({'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1','HF_DATASETS_OFFLINE':'1'})
build_deps=base/'kaggle-vllm-v030-py312-build-deps-v2'
base_wheels=[(build_deps/'jinja2-3.1.6-py3-none-any.whl','85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67'),(build_deps/'packaging-26.3-py3-none-any.whl','d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c'),(base/'kaggle-vllm-v030-py312-runtime-core-v2'/'markupsafe-3.0.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl','8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581')]
for path,expected in base_wheels: assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==expected,path
base_target=Path('/tmp/kaggle-vllm-v031-base-deps')
assert not base_target.exists(), 'Use a fresh Kaggle session'
subprocess.run([python312,'-m','pip','install','--no-index','--no-deps','--no-cache-dir','--target',str(base_target),*[str(path) for path,_ in base_wheels]],env=env,check=True,timeout=300)
env['PYTHONPATH']=os.pathsep.join((str(base_target),env['PYTHONPATH']))
sdk_target=Path('/tmp/kaggle-vllm-v031-experiment-sdk')
assert not sdk_target.exists(), 'Use a fresh Kaggle session'
subprocess.run([python312,'-m','pip','install','--no-index','--no-deps','--no-cache-dir','--target',str(sdk_target),str(wheel)],env=env,check=True,timeout=300)
env['PYTHONPATH']=os.pathsep.join((str(sdk_target),env['PYTHONPATH']))
bootstrap="from kaggle_vllm.l4_bootstrap import bootstrap; print(bootstrap(runtime_root='/kaggle/input/datasets/waqasm86', native_source='/kaggle/input/datasets/waqasm86/vllm-0-30-0-cp312-cp312-linux-x86-64/vllm-0.30.0-cp312-cp312-linux_x86_64.whl', strict=True))"
subprocess.run([python312,'-c',bootstrap],env=env,check=True,timeout=3600)
from pathlib import Path as _Path
runtime=_Path('/tmp/kaggle-vllm/v0.3.0/l4x4')
assert (runtime/'manifests/sdk-runtime.json').is_file()
env['PYTHONPATH']=os.pathsep.join((str(runtime/'pydeps'),env['PYTHONPATH'],str(sdk_input)))
env['LD_LIBRARY_PATH']=os.pathsep.join(filter(None,('/usr/local/nvidia/lib64',str(runtime/'pydeps/torch/lib'),env.get('LD_LIBRARY_PATH',''))))
env['LIBRARY_PATH']=os.pathsep.join(filter(None,(str(runtime/'cuda-driver-link'),env.get('LIBRARY_PATH',''))))
print('offline_runtime_ready',runtime)


In [ ]:
from pathlib import Path
import subprocess
script=sdk_input/'gemma_tp4_sharded_state_kaggle.py'
assert script.is_file()
output=Path('/kaggle/working/gemma-4-31b-it-qat-w4a16-ct-tp4-vllm030-sm89')
log=Path('/kaggle/working/04_tp4_sharded_state_research.log')
with log.open('w') as stream:
    result=subprocess.run([python312,str(script),'--model',str(model),'--output',str(output),'--sdk-wheel',str(wheel)],env=env,stdout=stream,stderr=subprocess.STDOUT,timeout=5400)
print('exit',result.returncode,'log_tail','\n'.join(log.read_text(errors='replace').splitlines()[-30:]))
assert result.returncode==0, 'TP4 export/reload did not validate; preserve the log and incomplete staging'
